# MultiASR-Colab-Subtitler
**三合一高速 AI 語音轉字幕工具（支援 YouTube 下載 / 本地音檔上傳 $\rightarrow$ 自動輸出 `.srt` / `.txt` / 時間戳逐字稿）**

本筆記本整合目前開源界最強的三大語音辨識引擎，採**動態按需安裝**與**自動顯存管理 (VRAM Cleanup)**，可在同一個 Colab 環境內隨時切換比較：
1. **Faster-Whisper (`large-v3-turbo` / `large-v3`)**：全能精準首選，內建 VAD 語音活動偵測，防止空白片段產生幻覺字幕。
2. **SenseVoice-Small (`FunASR`)**：阿里巴巴非自回歸極速模型，中英日韓粵混講表現極佳，推論速度極快。
3. **WhisperX (`large-v3-turbo`)**：結合 Wav2Vec2 強制音素對齊，提供毫秒級精準 SRT 時間軸。

> **使用提示**：請確認左上角選單 **「執行階段 (Runtime)」 $\rightarrow$ 「變更執行階段類型 (Change runtime type)」** 已設定為 **T4 GPU**。

In [ ]:
# @title 步驟 1：選擇音訊來源並進行標準化預處理
# @markdown 請選擇音訊來源（系統會自動透過 FFmpeg 轉換為標準 16kHz 單聲道 WAV，徹底解決 `.m4a` 或特殊編碼格式報錯問題）：

import os
import subprocess
import importlib.util
from google.colab import files

audio_source = "下載 YouTube 影片音訊" #@param ["下載 YouTube 影片音訊", "從電腦上傳新音檔", "手動輸入檔案路徑"]

# @markdown ---
# @markdown ### 選項 A：若選擇「下載 YouTube 影片音訊」，請貼上網址：
video_url = "https://www.youtube.com/watch?v=wSp6AiNIrsY" #@param {type:"string"}

# @markdown ### 選項 B：若選擇「手動輸入檔案路徑」，請填寫 Colab 內完整路徑：
custom_file_path = "/content/audio.m4a" #@param {type:"string"}

raw_audio_path = ""
PROCESSED_WAV_PATH = "/content/input_audio_16k.wav"

if audio_source == "下載 YouTube 影片音訊":
    if importlib.util.find_spec("yt_dlp") is None:
        print("[安裝] 正在安裝 yt-dlp...")
        subprocess.run(["pip", "install", "-q", "-U", "yt_dlp"], check=True)
    from yt_dlp import YoutubeDL

    print(f"[下載] 準備下載 YouTube 音訊: {video_url}")
    raw_audio_path = "/content/yt_downloaded_audio.m4a"
    if os.path.exists(raw_audio_path):
        os.remove(raw_audio_path)

    ydl_opts = {
        "overwrites": True,
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": raw_audio_path,
        "quiet": False,
    }
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])

elif audio_source == "從電腦上傳新音檔":
    print("[上傳] 請在下方點擊「選擇檔案」上傳您的音訊或影片檔...")
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        raw_audio_path = f"/content/{filename}"
        print(f"[完成] 成功上傳：{filename}")

elif audio_source == "手動輸入檔案路徑":
    raw_audio_path = custom_file_path.strip()

# --- 使用 FFmpeg 統一轉換為 16kHz 單聲道 WAV ---
if not raw_audio_path or not os.path.exists(raw_audio_path):
    print(f"[錯誤] 找不到檔案：{raw_audio_path}，請確認下載或上傳是否成功！")
else:
    print(f"[處理中] 正在將音訊標準化為 16kHz WAV 格式 ({raw_audio_path})...")
    ffmpeg_cmd = [
        "ffmpeg", "-y", "-i", raw_audio_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        PROCESSED_WAV_PATH
    ]
    res = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0 or not os.path.exists(PROCESSED_WAV_PATH):
        print("[錯誤] FFmpeg 音訊轉換失敗！錯誤訊息：")
        print(res.stderr[-500:])
    else:
        # 計算音訊長度
        import wave
        with wave.open(PROCESSED_WAV_PATH, "rb") as wf:
            duration_sec = wf.getnframes() / float(wf.getframerate())
        mins, secs = divmod(int(duration_sec), 60)
        print(f"[完成] 音訊準備完成！總長度：{mins} 分 {secs} 秒 ({PROCESSED_WAV_PATH})")
        print("請前往【步驟 2】選擇模型並開始語音辨識。")

In [ ]:
# @title 步驟 2：選擇語音辨識模型並生成字幕檔
# @markdown ### 1. 選擇語音轉錄引擎 (ASR Engine)：
asr_engine = "Faster-Whisper (large-v3-turbo)" #@param ["Faster-Whisper (large-v3-turbo)", "Faster-Whisper (large-v3)", "SenseVoice-Small (FunASR)", "WhisperX (large-v3-turbo)"]

# @markdown ### 2. 語言與後處理設定：
language = "auto" #@param ["auto", "zh", "en", "ja", "ko", "yue"]
convert_to_traditional_zh = True #@param {type:"boolean"}
vad_filter = True #@param {type:"boolean"}
auto_download_outputs = False #@param {type:"boolean"}

import os
import gc
import time
import subprocess
import importlib.util
import numpy as np
import torch
from google.colab import files

PROCESSED_WAV_PATH = "/content/input_audio_16k.wav"

def ensure_package(pkg_check_name, pip_install_args):
    if importlib.util.find_spec(pkg_check_name) is None:
        print(f"[安裝] 首次使用，正在安裝套件: {' '.join(pip_install_args)} ...")
        subprocess.run(["pip", "install", "-q", "-U"] + pip_install_args, check=True)

# 確保繁簡轉換工具已安裝
if convert_to_traditional_zh:
    ensure_package("opencc", ["opencc-python-reimplemented"])
    from opencc import OpenCC
    cc = OpenCC("s2twp")
else:
    cc = None

# 全域模型快取（避免同模型重複載入，並在切換模型時自動釋放 GPU 記憶體）
if "_CURRENT_ENGINE" not in globals():
    _CURRENT_ENGINE = None
    _MODEL_CACHE = None

def clear_gpu_memory():
    global _CURRENT_ENGINE, _MODEL_CACHE
    if _MODEL_CACHE is not None:
        print(f"[清理] 釋放前一個模型 ({_CURRENT_ENGINE}) 的 GPU 記憶體...")
        del _MODEL_CACHE
        _MODEL_CACHE = None
        _CURRENT_ENGINE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def format_srt_time(seconds):
    if seconds is None or seconds < 0:
        seconds = 0.0
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    ms = int(round((s - int(s)) * 1000))
    if ms >= 1000:
        ms = 999
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{ms:03d}"

def format_txt_time(seconds):
    if seconds is None or seconds < 0:
        seconds = 0.0
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if int(h) > 0:
        return f"{int(h):02d}:{int(m):02d}:{int(s):02d}"
    return f"{int(m):02d}:{int(s):02d}"

if not os.path.exists(PROCESSED_WAV_PATH):
    print(f"[錯誤] 找不到預處理音訊檔 {PROCESSED_WAV_PATH}，請先執行【步驟 1】！")
else:
    import wave
    start_wall = time.time()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    # 直接將 16kHz 單聲道 WAV 讀為 float32 NumPy 陣列，徹底繞過 PyAV (av.open) 版本不相容問題
    with wave.open(PROCESSED_WAV_PATH, "rb") as wf:
        sr = wf.getframerate()
        raw_pcm = wf.readframes(wf.getnframes())
        audio_np = np.frombuffer(raw_pcm, dtype=np.int16).astype(np.float32) / 32768.0

    if _CURRENT_ENGINE != asr_engine:
        clear_gpu_memory()

    segments_list = []

    # ==========================================================
    # 引擎 1 & 2：Faster-Whisper (large-v3-turbo / large-v3)
    # ==========================================================
    if asr_engine.startswith("Faster-Whisper"):
        ensure_package("faster_whisper", ["ctranslate2>=4.5.0", "faster-whisper>=1.1.0"])
        from faster_whisper import WhisperModel

        model_size = "large-v3-turbo" if "turbo" in asr_engine else "large-v3"
        if _MODEL_CACHE is None:
            print(f"[載入] 正在載入 Faster-Whisper ({model_size}) 至 {device.upper()}...")
            _MODEL_CACHE = WhisperModel(model_size, device=device, compute_type=compute_type)
            _CURRENT_ENGINE = asr_engine

        print("[辨識中] 開始進行語音辨識...")
        lang_arg = None if language == "auto" else language
        segments_gen, info = _MODEL_CACHE.transcribe(
            audio_np,
            language=lang_arg,
            beam_size=5,
            vad_filter=vad_filter,
            vad_parameters=dict(min_silence_duration_ms=500),
        )
        print(f"[資訊] 偵測語言: {info.language} (信心度: {info.language_probability:.2f})")
        for seg in segments_gen:
            txt = seg.text.strip()
            if txt:
                segments_list.append({"start": float(seg.start), "end": float(seg.end), "text": txt})

    # ==========================================================
    # 引擎 3：SenseVoice-Small (FunASR)
    # ==========================================================
    elif asr_engine.startswith("SenseVoice"):
        ensure_package("funasr", ["funasr>=1.1.12", "modelscope", "torchaudio", "soundfile"])
        from funasr import AutoModel
        from funasr.utils.postprocess_utils import rich_transcription_postprocess

        funasr_device = "cuda:0" if device == "cuda" else "cpu"
        if _MODEL_CACHE is None:
            print(f"[載入] 正在載入 SenseVoice-Small 與 FSMN-VAD 模型至 {funasr_device}...")
            asr_model = AutoModel(
                model="iic/SenseVoiceSmall",
                device=funasr_device,
                disable_update=True,
            )
            vad_model = AutoModel(
                model="fsmn-vad",
                vad_kwargs={"max_single_segment_time": 20000},
                device=funasr_device,
                disable_update=True,
            )
            _MODEL_CACHE = (asr_model, vad_model)
            _CURRENT_ENGINE = asr_engine

        asr_model, vad_model = _MODEL_CACHE
        print("[辨識中] 正在透過 FSMN-VAD 切分語音時間軸並執行 SenseVoice 極速轉錄...")
        vad_res = vad_model.generate(input=PROCESSED_WAV_PATH)
        vad_segments = vad_res[0].get("value", []) if vad_res else []

        if not vad_segments:
            vad_segments = [[0, int(len(audio_np) / sr * 1000)]]

        for start_ms, end_ms in vad_segments:
            start_sample = int(start_ms * sr / 1000)
            end_sample = int(end_ms * sr / 1000)
            chunk_audio = audio_np[start_sample:end_sample]
            if len(chunk_audio) < int(0.1 * sr):
                continue
            res = asr_model.generate(
                input=chunk_audio,
                cache={},
                language=language,
                use_itn=True,
                batch_size_s=60,
            )
            if res and res[0].get("text"):
                clean_txt = rich_transcription_postprocess(res[0]["text"]).strip()
                if clean_txt:
                    segments_list.append({
                        "start": start_ms / 1000.0,
                        "end": end_ms / 1000.0,
                        "text": clean_txt,
                    })

    # ==========================================================
    # 引擎 4：WhisperX (large-v3-turbo 強制時間軸對齊)
    # ==========================================================
    elif asr_engine.startswith("WhisperX"):
        ensure_package("whisperx", ["ctranslate2>=4.5.0", "faster-whisper>=1.1.0", "whisperx"])
        import whisperx

        if _MODEL_CACHE is None:
            print(f"[載入] 正在載入 WhisperX (large-v3-turbo) 至 {device.upper()}...")
            _MODEL_CACHE = whisperx.load_model("large-v3-turbo", device, compute_type=compute_type)
            _CURRENT_ENGINE = asr_engine

        print("[辨識中] 正在執行 WhisperX 批次轉錄...")
        lang_arg = None if language == "auto" else language
        result = _MODEL_CACHE.transcribe(audio_np, batch_size=16, language=lang_arg)
        detected_lang = result.get("language", "zh")

        try:
            print(f"[對齊中] 正在執行 Wav2Vec2 毫秒級時間軸強制對齊 (語言: {detected_lang})...")
            model_a, metadata = whisperx.load_align_model(language_code=detected_lang, device=device)
            result = whisperx.align(
                result["segments"], model_a, metadata, audio_np, device, return_char_alignments=False
            )
            del model_a
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[提示] 該語言不支援字級強制對齊或對齊略過 ({e})，將使用原始區段時間軸。")

        for seg in result.get("segments", []):
            txt = seg.get("text", "").strip()
            if txt:
                start_t = float(seg.get("start", 0.0))
                end_t = float(seg.get("end", start_t + 3.0))
                segments_list.append({"start": start_t, "end": end_t, "text": txt})

    # ==========================================================
    # 統一後處理（繁體化）與輸出三種檔案格式
    # ==========================================================
    srt_lines = []
    timestamped_lines = []
    plain_lines = []

    for idx, seg in enumerate(segments_list, start=1):
        start_t = seg["start"]
        end_t = seg["end"] if seg["end"] is not None else start_t + 5.0
        text_val = seg["text"].strip()
        if cc is not None:
            text_val = cc.convert(text_val)

        plain_lines.append(text_val)
        srt_lines.append(f"{idx}\n{format_srt_time(start_t)} --> {format_srt_time(end_t)}\n{text_val}\n")
        timestamped_lines.append(f"[{format_txt_time(start_t)} -> {format_txt_time(end_t)}] {text_val}")

    full_text = "\n".join(plain_lines)
    srt_content = "\n".join(srt_lines)
    timestamped_content = "\n".join(timestamped_lines)

    with open("/content/transcript.txt", "w", encoding="utf-8") as f:
        f.write(full_text)
    with open("/content/transcript.srt", "w", encoding="utf-8") as f:
        f.write(srt_content)
    with open("/content/timestamped.txt", "w", encoding="utf-8") as f:
        f.write(timestamped_content)

    elapsed = time.time() - start_wall
    print("=" * 50)
    print(f"[完成] 辨識完成！使用模型：{asr_engine} | 耗時：{elapsed:.1f} 秒 | 共 {len(segments_list)} 個字幕片段")
    print("=" * 50)
    print("辨識結果預覽（前 10 行）：")
    for line in timestamped_lines[:10]:
        print("  ", line)
    print("=" * 50)
    print("[輸出檔案] 已成功生成以下三個檔案：")
    print("  1. /content/transcript.txt   (純文字逐字稿)")
    print("  2. /content/transcript.srt   (標準 SRT 字幕檔)")
    print("  3. /content/timestamped.txt  (含時間戳記的閱讀稿)")

    if auto_download_outputs:
        print("[下載] 正在觸發瀏覽器自動下載...")
        files.download("/content/transcript.srt")
        files.download("/content/transcript.txt")
        files.download("/content/timestamped.txt")

In [ ]:
# @title 步驟 3：(選用) 掛載 Google Drive 並備份字幕檔
# @markdown 執行此區塊可將產出的三個檔案自動複製到您的 Google 雲端硬碟中：
import os
import shutil
from google.colab import drive

drive_folder = "/content/drive/MyDrive/MultiASR_Subtitles" #@param {type:"string"}

drive.mount("/content/drive")
os.makedirs(drive_folder, exist_ok=True)

for fname in ["transcript.txt", "transcript.srt", "timestamped.txt"]:
    src = f"/content/{fname}"
    if os.path.exists(src):
        dst = os.path.join(drive_folder, fname)
        shutil.copy2(src, dst)
        print(f"[完成] 已備份至雲端硬碟：{dst}")
    else:
        print(f"[提示] 找不到 {src}，請先執行【步驟 2】生成字幕。")